In [2]:
# === 依赖：scipy.spatial, plotly, numpy, pyomo, gurobi 可用 ===
# 把这段放在你现工程中（建议新文件），并确保已导入你贴出的
# build_models_from_csv / evaluate_Q_at / dump_solution 等工具。

import numpy as np
from scipy.spatial import Delaunay
import itertools as it
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
import plotly.graph_objects as go
import csv
from tqdm import tqdm



# ------------------------- 可调参数（与你确认的一致） -------------------------
MIN_DIST   = 1e-4     # 新点与已有点的最小距离
ACTIVE_TOL = 1e-8     # active 判定容差
MS_AGG     = "sum"    # 单形 ms 聚合口径：'sum'（与你现代码一致）；如需 'mean' 可改

# ------------------------- 工具：节点角点生成 -------------------------
def corners_from_var_bounds(vars_3):
    """从 3 个第一阶段变量的上下界生成 8 个角点"""
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    # 2^3 角点
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

# ------------------------- 工具：去重判定 -------------------------
def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

# ------------------------- 工具：打印表（列=四面体） -------------------------
def print_tetra_table(per_tet, active_mask, prec=6):
    # per_tet 按 simplex_index 升序
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}
    # 头
    header = ["row\\simp"] + [f"T{tid}{'*' if tid in active_set else ''}" for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       for r in per_tet],
    ]
    table = [header] + rows
    # 宽度
    colw = [0]*len(header)
    for c in range(len(header)):
        colw[c] = max(len(str(row[c])) for row in table) + 2
    # 颜色
    RED, RESET = "\033[31m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0: return s
        tid = tet_ids[col_idx-1]
        return f"{RED}{s}{RESET}" if tid in active_set else s
    # 打印
    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；第1行=UB，第2行=LB，第3行=ms)\n")

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    # 简洁打印：每个四面体一行（或分批），显示前若干场景 ms，避免爆屏
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]: 
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(6) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head)
    print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(6) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- 内层：单形×场景 的 ms 求解 -------------------------
def ms_on_tetra_for_scene(model_tmpl, first_vars, solver, tet_vertices, fverts_scene):
    """
    在一个四面体上，针对单个场景模型，求
      ms = min_{lambda>=0,1^T lambda=1} [ obj_expr(K(lambda)) - sum lambda_j f(v_j) ]
    其中 K(lambda) = sum lambda_j v_j （v_j ∈ R^3 是四个顶点），
    f(v_j) 为该场景在四个顶点上的真实值（已缓存）。
    返回：ms_value, lambda*, new_point(=K(lambda*))
    """
    # 克隆一个模型
    m = model_tmpl.clone()
    # 取新模型中的 Kp,Ki,Kd
    Kp = m.find_component(first_vars[0].name)
    Ki = m.find_component(first_vars[1].name)
    Kd = m.find_component(first_vars[2].name)
    if any(v is None for v in (Kp,Ki,Kd)):
        raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

    # 重心变量
    m.lam = pyo.Var(range(4), domain=pyo.NonNegativeReals)
    m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in range(4)) == 1.0)

    # 绑定 K = Σ λ v
    vx = [float(tet_vertices[j][0]) for j in range(4)]
    vy = [float(tet_vertices[j][1]) for j in range(4)]
    vz = [float(tet_vertices[j][2]) for j in range(4)]
    m.link_kp = pyo.Constraint(expr=Kp == sum(m.lam[j]*vx[j] for j in range(4)))
    m.link_ki = pyo.Constraint(expr=Ki == sum(m.lam[j]*vy[j] for j in range(4)))
    m.link_kd = pyo.Constraint(expr=Kd == sum(m.lam[j]*vz[j] for j in range(4)))

    # As = Σ λ f(v_j) （该场景）
    m.As = pyo.Var()
    m.As_def = pyo.Constraint(expr=m.As == sum(m.lam[j]*float(fverts_scene[j]) for j in range(4)))

    # 目标：min obj_expr - As
    # 注意：obj_expr 已在 build_pid_model 中定义；这里直接用
    if hasattr(m, 'obj'): m.del_component('obj')
    m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

    # 解
    # 可用 file-based gurobi，也可 persistent。这里直接用 file-based 你上面配置的 solver
    res = solver.solve(m, tee=False)
    ok = (res.solver.status == SolverStatus.ok) and \
         (res.solver.termination_condition == TerminationCondition.optimal)
    if not ok:
        # 容错：返回一个偏保守的大 ms（也可抛异常）
        # 这里返回 +inf 让该四面体在排序中被忽略
        return float('inf'), None, None

    ms_val = pyo.value(m.obj)
    lam_star = np.array([pyo.value(m.lam[j]) for j in range(4)], dtype=float)
    new_pt = np.dot(lam_star, np.array(tet_vertices, dtype=float))
    return float(ms_val), lam_star, tuple(map(float, new_pt))

# ------------------------- 主流程：一轮评估所有四面体 -------------------------
def evaluate_all_tetra(nodes, scen_values, model_list, first_vars_list, solver):
    """
    nodes:         list of 3D points (current nodes)
    scen_values:   list (len=S) of lists f_ω(node_i) 已缓存；形状 S × len(nodes)
    model_list:    场景模型列表
    first_vars_list: 场景中第一阶段变量列表 [ [Kp,Ki,Kd], ... ]
    solver:        Gurobi solver (file-based)
    返回 tri, per_tet 结构：
      per_tet = [{
        'simplex_index': k,
        'vert_idx': [i0,i1,i2,i3],
        'verts': [(..),(..),(..),(..)],
        'fverts_sum': [sum over ω f_ω(v_j)],  # 4 个顶点的“总 f”值
        'ms_per_scene': [ms_{ω}] (len=S),
        'ms': sum(ms_{ω}),
        'LB': min_j sum_ω f_ω(v_j) + sum_ω ms_{ω},
        'UB': max_j sum_ω f_ω(v_j) + sum_ω ms_{ω},
        'x_ms_best_scene': argmin_ω 的 new_point,
      }, ...]
    """
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []

    tri = Delaunay(pts)  # 3D: simplices shape = (M, 4)
    S = len(model_list)

    # 预先把每个顶点的“总 f 值”算好
    scen_values = [list(arr) for arr in scen_values]  # 确保可索引
    f_sum_per_node = [sum(scen_values[ω][i] for ω in range(S)) for i in range(len(nodes))]

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]
        # 各场景在四个顶点的 f
        fverts_per_scene = [
            [scen_values[ω][i] for i in idxs] for ω in range(S)
        ]
        # “总 f”顶点向量（用于 LB/UB 中的 min/max）
        fverts_sum = [sum(fverts_per_scene[ω][j] for ω in range(S)) for j in range(4)]

        # 逐场景 ms
        ms_scene = []
        xms_scene = []
        for ω in range(S):
            ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                model_list[ω], first_vars_list[ω], solver, verts, fverts_per_scene[ω]
            )
            ms_scene.append(ms_val)
            xms_scene.append(new_pt)

        # 聚合 ms（与你现代码一致：sum；如果想用平均，可改为 np.mean）
        if MS_AGG == "sum":
            ms_total = float(np.sum(ms_scene))
        elif MS_AGG == "mean":
            ms_total = float(np.mean(ms_scene))
        else:
            raise ValueError("MS_AGG must be 'sum' or 'mean'")

        LB = float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + ms_total)

        # 为选点：取该四面体中“ms 最小”的场景的 new_pt（与现代码风格一致）
        best_scene = int(np.argmin(ms_scene))
        x_ms_best = xms_scene[best_scene]

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,
            "ms_per_scene": ms_scene,
            "ms": ms_total,
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms_best,
            "best_scene": best_scene,
        })

    return tri, per_tet

# ------------------------- 可视化（Plotly） -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet):
    """绘制节点+active 四面体边；质心 hover 显示单形信息"""
    fig = go.Figure()

    nodes = np.asarray(nodes, float)
    # 所有节点 - 黑点
    fig.add_trace(go.Scatter3d(
        x=nodes[:,0], y=nodes[:,1], z=nodes[:,2],
        mode='markers',
        marker=dict(size=4),
        name='nodes'
    ))

    # 绿色点：当前 UB 节点（全局最小值节点）
    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=6, symbol="circle", color="green"),
            name='current min node'
        ))

    # 蓝色点：下一轮新节点
    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=6, symbol="circle", color="blue"),
            name='next node'
        ))

    # 画 active 四面体的边（深灰线）
    if tri is not None:
        pts = tri.points
        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue
            idxs = r["vert_idx"]
            # 四面体的 6 条边
            edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
            for (a,b) in edges:
                pa = pts[idxs[a]]; pb = pts[idxs[b]]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=4, color="gray"),
                    name='active edge',
                    showlegend=False
                ))
        # 质心 hover（不可见散点）
        cent_x, cent_y, cent_z, texts = [], [], [], []
        for r in per_tet:
            sid = r["simplex_index"]
            v = np.mean(np.asarray(r["verts"]), axis=0)
            ms_strs = [f"{v:.2e}" for v in r["ms_per_scene"][:8]]
            more = "" if len(r["ms_per_scene"])<=8 else f" (+{len(r['ms_per_scene'])-8} more)"
            txt = (f"simp={sid}<br>"
                   f"LB={r['LB']:.6f}<br>UB={r['UB']:.6f}<br>"
                   f"ms={r['ms']:.3e}<br>"
                   f"ms_per_scene: [{', '.join(ms_strs)}]{more}")
            cent_x.append(v[0]); cent_y.append(v[1]); cent_z.append(v[2]); texts.append(txt)
        fig.add_trace(go.Scatter3d(
            x=cent_x, y=cent_y, z=cent_z,
            mode='markers',
            marker=dict(size=1, opacity=0.0),
            text=texts, hoverinfo="text",
            name="tetra info", showlegend=False
        ))

    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube"
        ),
        width=800, height=600
    )
    fig.show()

# ------------------------- 主循环 -------------------------
def run_pid_simplex_3d(
    model_list, first_vars_list, solver, target_nodes=30,
    min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True
):
    """
    model_list:        场景模型列表
    first_vars_list:   每个模型的 [Kp,Ki,Kd]
    solver:            SolverFactory('gurobi') 已设置 NonConvex=2 等参数
    target_nodes:      目标节点数
    返回：历史等
    """
    S = len(model_list)
    # 初始节点：8 个角点
    start_nodes = corners_from_var_bounds(first_vars_list[0])
    nodes = list(start_nodes)

    # 缓存：场景-节点真值 f_ω(node_i)
    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = float(evaluate_Q_at(model_list[ω], first_vars_list[ω], node, solver))

    # 历史
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []

    it = 0
    while len(nodes) < target_nodes:
        # 当前全局 UB & UB节点
        f_sum_per_node = [sum(scen_values[ω][i] for ω in range(S)) for i in range(len(nodes))]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 三角剖分（四面体）并逐单形评估
        tri, per_tet = evaluate_all_tetra(nodes, scen_values, model_list, first_vars_list, solver)
        if tri is None or not per_tet:
            if verbose: print("Not enough nodes to make tetrahedra; stop.")
            break

        # active mask
        active_mask = { r["simplex_index"]: (r["LB"] <= UB_global + active_tol) for r in per_tet }
        active = [r for r in per_tet if active_mask[r["simplex_index"]]]

        # 全局 LB（你新口径）：UB 节点相邻单形的 ms 最大值 + UB 节点值
        ms_adj = [r["ms"] for r in per_tet if ub_idx in r["vert_idx"]]
        if ms_adj:
            LB_global = UB_global + float(np.max(ms_adj))
        else:
            # 理论上不该为空，兜底：取 min(LB_Δ)
            LB_global = float(np.min([r["LB"] for r in per_tet]))

        # 记录 ms 指标：本轮所有单形 ms 的最小值（延续你原先图二）
        ms_iter = float(np.min([r["ms"] for r in per_tet]))

        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(ms_iter)
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)

        # 打印表
        print_tetra_table(per_tet, active_mask)
        print_per_scenario_ms(per_tet, max_scenarios_to_print=10)
        print(f"[Global UB] = {UB_global:.6f}\n")

        # 选新点：仅从 active（若为空则退回全部）
        candidates = active if len(active) > 0 else per_tet
        best = min(candidates, key=lambda r: r["ms"])
        new_node = best["x_ms_best_scene"]
        if new_node is None or too_close(new_node, nodes, tol=min_dist):
            if verbose:
                print("New node too close (or infeasible ms); stop.")
            break

        # 可视化（Plotly）
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet)

        # 真函数评估（所有场景）
        new_vals = []
        for ω in range(S):
            val = float(evaluate_Q_at(model_list[ω], first_vars_list[ω], new_node, solver))
            new_vals.append(val)

        # 加入节点与缓存更新
        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist
    }



# ------------------------- 使用示例 -------------------------
#if __name__ == "__main__":
# ===================== MAIN =====================

# —— 运行规模：二选一（先用 Quick Test 更稳） ——
RUN_QUICK_TEST = True   # True: 小规模快速验证；False: 全量

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 5          # 先用少量场景测试链路
    target_nodes   = 10         # 先跑少量节点
else:
    csv_path       = "data.csv"
    max_scenarios  = 99         # 你的全量设置
    target_nodes   = 30

# 变量边界（与你之前一致）
bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 10),
    "Ki": (0, 10),
    "Kd": (0, 10),
}

# 构建多场景 PID 模型
weights = (1.0, 0.01)
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# Gurobi（file-based）
solver = pyo.SolverFactory('gurobi')
solver.options.update({
    'MIPGap': 1e-2,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    # 'TimeLimit': 30,  # 可选
})

# 运行 3D 单形法（四面体）
hist = run_pid_simplex_3d(
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    solver=solver,
    target_nodes=target_nodes,
    min_dist=1e-4,
    active_tol=1e-8,
    verbose=True
)

# （可选）简单检查输出
print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")



NameError: name 'build_models_from_csv' is not defined

In [ ]:
# ==== 环境依赖 ====
# pip install pyomo scipy numpy
# 并准备一个非线性连续求解器（建议 IPOPT）。若无 IPOPT，可先试 Couenne/Bonmin 或者把目标改成能被二阶锥/二次求解的形式。

import numpy as np
from scipy.spatial import Delaunay
import itertools as it
import pyomo.environ as pyo
from pyomo.contrib.piecewise import PiecewiseLinearFunction
from pyomo.core.base import TransformationFactory
import matplotlib.pyplot as plt
from matplotlib.projections import register_projection
from mpl_toolkits.mplot3d import Axes3D
register_projection(Axes3D)            
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib
matplotlib.use("TkAgg")   # 或 "TkAgg"；二选一，电脑里哪个可用就用哪个
#%matplotlib widget
import numpy as np
from scipy.spatial import Delaunay
import itertools as it
import pyomo.environ as pyo
# pip install pyomo scipy numpy
# 建议安装 IPOPT；若没有，可把 solver 改成 'bonmin' 或 'couenne' 试跑

import numpy as np
import itertools as it
from scipy.spatial import Delaunay
import pyomo.environ as pyo
# pip install pyomo scipy numpy
# 建议 solver 用 IPOPT；若没有，可尝试 'bonmin' 或 'couenne'
import numpy as np
import itertools as it
from scipy.spatial import Delaunay
import pyomo.environ as pyo
